# Spin-1 XY Sec. VI P0 provisioning companion

This restartable companion implements the current Sec. VI provisioning handoff without
re-running the already-converged 10000-eigenpair calculation.  It uses the locked
representative Hamiltonian

\[
H_\kappa=K_1(J)+K_3(0.1J)+K_2(i\kappa),\qquad \kappa_\star/J=0.1,
\]

with even periodic chains at fixed $M=-2$ in the tower momentum sector.  The workflow is
organized around the draft's P0.0--P0.3 requests:

- **P0.0:** repair the existing sparse-convergence table and checkpoint every newly required
  sparse spectrum immediately after the eigensolve, before covariance/fitting/rendering;
- **P0.1:** compute the complete 19-operator block-invariant two-site covariance at $L=14$
  in the contained $\Delta E=1$ window, together with raw/clean companions, residual and
  exact-energy-block audits;
- **P0.2:** keep the raw microcanonical sequence primary and explicitly split local ensemble
  equivalence into
  $\rho_{\rm mc}^{(M,k)}\leftrightarrow\rho_{\beta=0}^{(M,k)}
  \leftrightarrow\rho_{\beta=0}^{M}$, resolving the residual operator in the same
  Hilbert--Schmidt-normalized 19-dimensional algebra;
- **P0.3:** optionally add the nonrepresentative $L=14$, $\kappa/J=0.20$ family point only
  after the representative concentration job is secure.

The expensive spectral checkpoint is deliberately a separate artifact from the executed
notebook, so a later pandas or plotting failure can be resumed without another large solve.


## Parameters and evidence roots


In [ ]:
import sys
from pathlib import Path

from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
JOBS_DIR = ROOT / "experimental" / "jobs"
NOTEBOOK_DIR = ROOT / "experimental" / "notebooks"
for path in (JOBS_DIR, NOTEBOOK_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

RUN_PROFILE = "smoke"
USE_TEX = False
FIGURE_FORMATS = ("pdf", "svg")
DATA_DIR = ROOT / "experimental" / "data" / "spin1_xy_sec6_provisioning"
EVIDENCE_ROOT = ROOT / "experimental" / "data" / "evidence_jobs"
BASELINE_DATA_DIR = EVIDENCE_ROOT / "spin1_production_20260806T074051Z"
SPARSE_CONVERGENCE_DATA_DIR = EVIDENCE_ROOT / "spin1_production_20260810T082123Z"
CHECKPOINT_SOURCE_DIR = None
CHECKPOINT_DIR = None

DENSE_SIZES = (8,) if RUN_PROFILE == "smoke" else (8, 10, 12)
LARGE_SIZE = 14
REPRESENTATIVE_KAPPA_OVER_J = 0.10
FAMILY_KAPPA_OVER_J = 0.20
REPRESENTATIVE_EIGENPAIRS = 8192
FAMILY_EIGENPAIRS = 8192
SAFE_FIXED_HALF_WIDTHS = (0.75, 1.0, 1.25)
INCLUDE_QUARTER_WINDOW = True
CONCENTRATION_HALF_WIDTH = 1.0
SHIFT = 1.0e-7
ARPACK_TOLERANCE = 1.0e-9
RESIDUAL_CHUNK_SIZE = 64
REUSE_CHECKPOINTS = True
WRITE_CHECKPOINTS = True
RUN_LARGE_REPRESENTATIVE = RUN_PROFILE == "production"
RUN_FAMILY_LARGE_SIZE = False

DATA_DIR.mkdir(parents=True, exist_ok=True)
print("profile:", RUN_PROFILE)
print("output:", DATA_DIR)
print("dense sizes:", DENSE_SIZES)
print("large representative:", RUN_LARGE_REPRESENTATIVE)
print("family L=14 follow-up:", RUN_FAMILY_LARGE_SIZE)

## Execute restartable P0 provisioning


In [ ]:
from spin1_sec6_provisioning import Sec6ProvisioningConfig
from spin1_sec6_provisioning_contract import run_sec6_provisioning

config = Sec6ProvisioningConfig(
    output_dir=DATA_DIR,
    baseline_data_dir=BASELINE_DATA_DIR,
    sparse_convergence_data_dir=SPARSE_CONVERGENCE_DATA_DIR,
    checkpoint_source_dir=CHECKPOINT_SOURCE_DIR,
    checkpoint_dir=CHECKPOINT_DIR,
    dense_sizes=tuple(int(value) for value in DENSE_SIZES),
    large_size=int(LARGE_SIZE),
    representative_kappa_over_j=float(REPRESENTATIVE_KAPPA_OVER_J),
    family_kappa_over_j=float(FAMILY_KAPPA_OVER_J),
    representative_eigenpairs=int(REPRESENTATIVE_EIGENPAIRS),
    family_eigenpairs=int(FAMILY_EIGENPAIRS),
    fixed_half_widths=tuple(float(value) for value in SAFE_FIXED_HALF_WIDTHS),
    include_quarter_window=bool(INCLUDE_QUARTER_WINDOW),
    concentration_half_width=float(CONCENTRATION_HALF_WIDTH),
    shift=float(SHIFT),
    arpack_tolerance=float(ARPACK_TOLERANCE),
    residual_chunk_size=int(RESIDUAL_CHUNK_SIZE),
    reuse_checkpoints=bool(REUSE_CHECKPOINTS),
    write_checkpoints=bool(WRITE_CHECKPOINTS),
    run_large_representative=bool(RUN_LARGE_REPRESENTATIVE),
    run_family_large_size=bool(RUN_FAMILY_LARGE_SIZE),
)
products = run_sec6_provisioning(config)

## Claim-oriented inspection


In [ ]:
for name in (
    "sparse_convergence",
    "concentration_L14",
    "bridge_distances",
    "microcanonical_fits",
    "bridge_fits",
    "family_matching",
):
    frame = products[name]
    print(f"\n{name}: {len(frame)} rows")
    if not frame.empty:
        display(frame)

print("\nPrimary interpretation guardrails:")
print("- raw microcanonical values are the defining ETH sequence; cleaned values are diagnostic")
print("- beta=0 fixed-M limits are auxiliary until both local bridges close")
print("- L=14 covariance is shown only when the sparse budget audit is certified")
print("- no L=16 calculation is scheduled by this notebook")

## Export contract

The workflow writes the requested draft-facing products directly into `DATA_DIR`, including
`spin1_xy_kappa0p1_concentration_L14.csv`, the updated concentration sequence and fit,
`spin1_xy_kappa0p1_two_bridge_rdm_distance.csv`,
`spin1_xy_kappa0p1_residual_operator_spectrum.csv`, and
`spin1_xy_kappa0p1_residual_operator_coefficients.csv`.  The existing shared Spin-1 figure
renderer consumes the copied baseline grids plus the new L=14 representative/family products,
so Fig. 6 can be rendered as a separate lightweight stage after numerical completion.
